In [0]:
# CELL 0 — INSTALL DEPENDENCIES
%pip install yfinance

# This restarts the Python kernel automatically after installing
# Everything below this cell will work fine
dbutils.library.restartPython()

In [0]:
# CELL 1 — IMPORTS
import yfinance as yf
import pandas as pd
import os
from datetime import datetime

print("Libraries loaded")

In [0]:
# Paste this at the TOP of Cell 5 before the loop
# In case earlier cells didn't run

import yfinance as yf
import pandas as pd
import os
import time
from datetime import datetime

BASE_PATH = "/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/yfinance"

tickers = [
    "SHEL.L", "HSBA.L", "BP.L", "ULVR.L", "VOD.L",
    "BARC.L", "AZN.L", "SGE.L", "GRG.L", "BWY.L",
    "RSW.L", "PNN.L", "SVT.L", "IMI.L", "BBY.L",
    "DRX.L", "TEP.L", "CNA.L", "NCC.L", "SBRY.L"
]

ticker_meta = {
    "SHEL.L": "Shell plc",
    "HSBA.L": "HSBC Holdings plc",
    "BP.L":   "BP p.l.c.",
    "ULVR.L": "Unilever PLC",
    "VOD.L":  "Vodafone Group Plc",
    "BARC.L": "Barclays PLC",
    "AZN.L":  "AstraZeneca PLC",
    "SGE.L":  "The Sage Group plc",
    "GRG.L":  "Greggs plc",
    "BWY.L":  "Bellway plc",
    "RSW.L":  "Renishaw plc",
    "PNN.L":  "Pennon Group plc",
    "SVT.L":  "Severn Trent plc",
    "IMI.L":  "IMI plc",
    "BBY.L":  "Balfour Beatty plc",
    "DRX.L":  "Drax Group plc",
    "TEP.L":  "Telecom Plus plc",
    "CNA.L":  "Centrica plc",
    "NCC.L":  "NCC Group plc",
    "SBRY.L": "J Sainsbury plc",
}

DATASETS = ["income_statement", "balance_sheet", "cashflow", "history", "stats"]

FOLDERS = {}
for d in DATASETS:
    path = f"{BASE_PATH}/{d}"
    os.makedirs(path, exist_ok=True)
    FOLDERS[d] = path

def enrich(df, ticker):
    df = df.copy()
    df["ticker"]       = ticker
    df["company_name"] = ticker_meta.get(ticker, "Unknown")
    df["ingestion_ts"] = datetime.now().isoformat()
    df["source"]       = "yfinance"
    return df

print("All variables ready — starting loop...")

In [0]:
# CELL 3 — FOLDER SETUP
# Creates a dated folder for each dataset type:
#   /Volumes/.../yfinance/history/2026/06/06/
#   /Volumes/.../yfinance/income_statement/2026/06/06/
#   etc.

now   = datetime.now()
YEAR  = now.strftime("%Y")   # "2026"
MONTH = now.strftime("%m")   # "06"
DAY   = now.strftime("%d")   # "06"

DATASETS = [
    "income_statement",
    "balance_sheet",
    "cashflow",
    "history",
    "stats",
]

FOLDERS = {}
for d in DATASETS:
    path = f"{BASE_PATH}/{d}/{YEAR}/{MONTH}/{DAY}"
    os.makedirs(path, exist_ok=True)
    FOLDERS[d] = path
    print(f"Ready: {path}")

In [0]:
# CELL 4 — HELPER FUNCTIONS

def enrich(df, ticker):
    """
    Adds metadata columns to every dataframe before saving.
    This means every file knows where it came from and when.
    """
    df = df.copy()
    df["ticker"]       = ticker
    df["company_name"] = TICKER_META.get(ticker, "Unknown")
    df["ingestion_ts"] = datetime.now().isoformat()
    df["source"]       = "yfinance"
    return df


def save_df(df, path, ticker, orient="index"):
    """
    Saves a dataframe to JSON.
    Converts column names to strings first — yfinance uses
    Timestamps as column names which break JSON serialisation.
    """
    # yfinance uses Timestamp objects as column names after .T
    # We convert them to strings so JSON doesn't crash
    df.columns = [str(c) for c in df.columns]
    df.to_json(path, orient=orient, date_format="iso")
    print(f"  [{ticker}] saved {os.path.basename(path)}"
          f" — {len(df)} rows, {len(df.columns)} cols")


def safe_fetch(func, ticker, label):
    """
    Wraps any yfinance call in a try/except.
    Returns the result or None — never crashes the loop.
    
    func  = a lambda that does the actual yfinance call
    label = what we're fetching, for the error message
    """
    try:
        result = func()
        if result is None or (hasattr(result, "empty") and result.empty):
            print(f"  [{ticker}] {label}: no data returned")
            return None
        return result
    except Exception as e:
        print(f"  [{ticker}] {label} ERROR: {e}")
        return None


print("Functions ready")

In [0]:
# CELL 5 — FIXED MAIN LOOP
for t in tickers:
    print(f"\nProcessing: {t}")
    stock = yf.Ticker(t)

    # Income Statement
    try:
        income = stock.financials.T
        income = enrich(income, t)
        income.to_json(f"{FOLDERS['income_statement']}/{t}.json", orient="index")
        print(f"  income saved")
    except Exception as e:
        print(f"  income FAILED: {e}")

    # Balance Sheet
    try:
        balance = stock.balance_sheet.T
        balance = enrich(balance, t)
        balance.to_json(f"{FOLDERS['balance_sheet']}/{t}.json", orient="index")
        print(f"  balance saved")
    except Exception as e:
        print(f"  balance FAILED: {e}")

    # Cashflow
    try:
        cashflow = stock.cashflow.T
        cashflow = enrich(cashflow, t)
        cashflow.to_json(f"{FOLDERS['cashflow']}/{t}.json", orient="index")
        print(f"  cashflow saved")
    except Exception as e:
        print(f"  cashflow FAILED: {e}")

    # History
    try:
        history = stock.history(period="5y")
        history = enrich(history, t)
        history.to_json(f"{FOLDERS['history']}/{t}.json")
        print(f"  history saved — {len(history)} rows")
    except Exception as e:
        print(f"  history FAILED: {e}")

    # Stats
    try:
        stats = pd.DataFrame([stock.info])
        stats = enrich(stats, t)
        stats.to_json(f"{FOLDERS['stats']}/{t}.json", orient="records")
        print(f"  stats saved")
    except Exception as e:
        print(f"  stats FAILED: {e}")

    time.sleep(1)

print("\nyfinance ingestion complete")

In [0]:
# Temporary debug cell — run this for ONE ticker first
import yfinance as yf

t = "SHEL.L"
stock = yf.Ticker(t)

print("Testing each fetch one by one...\n")

# Test 1 — income
try:
    income = stock.financials
    print(f"income shape: {income.shape}")
    print(income.head(2))
except Exception as e:
    print(f"income FAILED: {e}")

# Test 2 — balance
try:
    balance = stock.balance_sheet
    print(f"\nbalance shape: {balance.shape}")
    print(balance.head(2))
except Exception as e:
    print(f"balance FAILED: {e}")

# Test 3 — cashflow
try:
    cashflow = stock.cashflow
    print(f"\ncashflow shape: {cashflow.shape}")
    print(cashflow.head(2))
except Exception as e:
    print(f"cashflow FAILED: {e}")

# Test 4 — history
try:
    history = stock.history(period="5y")
    print(f"\nhistory shape: {history.shape}")
    print(history.head(2))
except Exception as e:
    print(f"history FAILED: {e}")

# Test 5 — stats
try:
    info = stock.info
    print(f"\nstats keys: {list(info.keys())[:5]}")
except Exception as e:
    print(f"stats FAILED: {e}")

In [0]:
# CELL 6 — SUMMARY
print("\n" + "="*75)
print("  YFINANCE INGESTION SUMMARY")
print("="*75)
print(f"  {'#':>2}  {'Company':<28}  {'IS':^4}  {'BS':^4}  {'CF':^4}  {'HX':^4}  {'ST':^4}")
print(f"  {'':>2}  {'':28}  {'Inc':^4}  {'Bal':^4}  {'Csh':^4}  {'Hst':^4}  {'Inf':^4}")
print("-"*75)

issues = []
for i, r in enumerate(summary, 1):
    def flag(v): return "OK" if v else "FAIL"
    
    row = (f"  {i:>2}  {r['company']:<28}  "
           f"{flag(r['income_statement']):^4}  "
           f"{flag(r['balance_sheet']):^4}  "
           f"{flag(r['cashflow']):^4}  "
           f"{flag(r['history']):^4}  "
           f"{flag(r['stats']):^4}")
    print(row)

    if not all([r['income_statement'], r['balance_sheet'],
                r['cashflow'], r['history'], r['stats']]):
        issues.append(r['ticker'])

print("="*75)
print(f"  Saved to: {BASE_PATH}")
print(f"  Date partition: {YEAR}/{MONTH}/{DAY}")
if issues:
    print(f"  Partial failures: {', '.join(issues)}")
else:
    print(f"  All {len(summary)} tickers complete with no errors")
print("="*75)

old code 

In [0]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime

# ---------------------------------------------------
# Tickers (UK equities)
# ---------------------------------------------------
tickers = [
    "SHEL.L", "HSBA.L", "BP.L", "ULVR.L", "VOD.L",
    "BARC.L", "AZN.L", "SGE.L", "GRG.L", "BWY.L",
    "RSW.L", "PNN.L", "SVT.L", "IMI.L", "BBY.L",
    "DRX.L", "TEP.L", "CNA.L", "NCC.L", "SBRY.L"
]

# ---------------------------------------------------
# Company metadata (dimension)
# ---------------------------------------------------

ticker_meta = {
    "SHEL.L": "Shell plc",
    "HSBA.L": "HSBC Holdings plc",
    "BP.L": "BP p.l.c.",
    "ULVR.L": "Unilever PLC",
    "VOD.L": "Vodafone Group Plc",
    "BARC.L": "Barclays PLC",
    "AZN.L": "AstraZeneca PLC",
    "SGE.L": "The Sage Group plc",
    "GRG.L": "Greggs plc",
    "BWY.L": "Bellway plc",
    "RSW.L": "Renishaw plc",
    "PNN.L": "Pennon Group plc",
    "SVT.L": "Severn Trent plc",
    "IMI.L": "IMI plc",
    "BBY.L": "Balfour Beatty plc",
    "DRX.L": "Drax Group plc",
    "TEP.L": "Telecom Plus plc",
    "CNA.L": "Centrica plc",
    "NCC.L": "NCC Group plc",
    "SBRY.L": "J Sainsbury plc"
}

# ---------------------------------------------------
# Time partition (snapshot date)
# ---------------------------------------------------
now = datetime.now()
year = now.strftime("%Y")
month = now.strftime("%m")
day = now.strftime("%d")

BASE_PATH = "/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/yfinance"

# ---------------------------------------------------
# Dataset folders
# ---------------------------------------------------
datasets = [
    "income_statement",
    "balance_sheet",
    "cashflow",
    "history",
    "stats"
]

# ---------------------------------------------------
# Create folder structure
# ---------------------------------------------------
for d in datasets:
    os.makedirs(f"{BASE_PATH}/{d}/{year}/{month}/{day}", exist_ok=True)

# ---------------------------------------------------
# Helper function to enrich dataframe
# ---------------------------------------------------
def enrich(df, ticker):
        df = df.copy()
        df["ticker"] = ticker
        df["company_name"] = ticker_meta.get(ticker, "Unknown")
        df["ingestion_ts"] = datetime.now()
        df["source"] = "yfinance"
        return df


# ---------------------------------------------------
# Ingestion loop
# ---------------------------------------------------

for t in tickers:
    print("Processing:", t)

    stock = yf.Ticker(t)

    # ---------------- Income Statement ----------------
    try:
        income = stock.financials.T
        income = enrich(income, t)
        income.to_json(
            f"{BASE_PATH}/income_statement/{year}/{month}/{day}/{t}.json",
            orient="index"
        )
    except Exception as e:
        print("Income error:", t, e)

    # ---------------- Balance Sheet ----------------
    try:
        balance = stock.balance_sheet.T
        balance = enrich(balance, t)
        balance.to_json(
            f"{BASE_PATH}/balance_sheet/{year}/{month}/{day}/{t}.json",
            orient="index"
        )
    except Exception as e:
        print("Balance error:", t, e)

    # ---------------- Cashflow ----------------
    try:
        cashflow = stock.cashflow.T
        cashflow = enrich(cashflow, t)
        cashflow.to_json(
            f"{BASE_PATH}/cashflow/{year}/{month}/{day}/{t}.json",
            orient="index"
        )
    except Exception as e:
        print("Cashflow error:", t, e)

    # ---------------- History ----------------
    try:
        history = stock.history(period="5y")
        history = enrich(history, t)
        history.to_json(
            f"{BASE_PATH}/history/{year}/{month}/{day}/{t}.json"
        )
    except Exception as e:
        print("History error:", t, e)

    # ---------------- Stats ----------------
    try:
        stats = pd.DataFrame([stock.info])
        stats = enrich(stats, t)
        stats.to_json(
            f"{BASE_PATH}/stats/{year}/{month}/{day}/{t}.json",
            orient="records"
        )
    except Exception as e:
        print("Stats error:", t, e)

print("\nYFinance ingestion complete")